In [1]:
from utils import *
from prepare_ds import *
from train_ml import *
from create_map import *
from validation import *
from visualisation import *
from tqdm import tqdm

force = False
downgrade_labels = force

if downgrade_labels:
    src = DEFAULT_PATH["labels"]
    out = DEFAULT_PATH["labels"] + "downgraded/"
    classes_matching = DEFAULT_PATH["labels"] + "classes_matching.csv"
    downgrade_classes(src, out, classes_matching, force=True)
    DEFAULT_PATH["labels"] = out

# you can use force=True for re-write all caches.
signs = parse_tifs_from(DEFAULT_PATH["images"], typeof="sign", force=force)
signs = signs.query("type == 'sign'")
labels = parse_tifs_from(DEFAULT_PATH["labels"], "label", force=force)
labels = labels.query("type == 'label'")

# Prepare data for generate_dataset.
year = 2023
# only_bands = ["r", "b", "g", "n"]

# signs_paths = (
#     signs.query(f"year == {year} and season == 'mon' and band in @only_bands")
#     .sort_values("month")
#     .sort_values("band")
# )
labels_paths = labels.query(f"year == {year}")


# for 10m low RAM load
only_bands = ["r", "b", "g", "n", "swir1"]
tiles = DEFAULT_PATH["images"] + "tiles/"
tiles = parse_tifs_from(tiles, typeof="tile", force=force)
tiles = (
    tiles.query(f"year == {year} and band in @only_bands")
    .sort_values("month")
    .sort_values("band")
)
signs_paths = tiles

Initialization paths...
All paths was initialized.
Load 'sign' from cached DataFrame to path:  /Users/stephenhawking/Coding/ML/low2high_map/src/../data/processing/path2tif_sign.csv
Load 'label' from cached DataFrame to path:  /Users/stephenhawking/Coding/ML/low2high_map/src/../data/processing/path2tif_label.csv
Load 'tile' from cached DataFrame to path:  /Users/stephenhawking/Coding/ML/low2high_map/src/../data/processing/path2tif_tile.csv


In [4]:
config_grid = {
    # "resize": ["by_sign"],
    "resize": ["all_signs"],
    "mask_mode": ["random", "secure"],
    "layer_mode": [None],
    "layer_type": ["static"],
    # "r": [0.25 * i for i in range(6)[1:]],
    "r": [20],
    "count_signs": [3000000000],
    # "percent": [0.01 * i for i in range(6)][1:],
    # "homogen_percent": [0.1 * i for i in range(5)][2:],
    "homogen_percent": [0.1],
    # "stratify": [True, False],
    "stratify": [True],
}
best_models = analyse_best_model(
    config_grid,
    signs_paths,
    labels_paths,
    store_models=True,
    verbose=False,
    skip_train=True,
)

take_first = len(best_models)
models = []
for m in best_models[:take_first]:
    models.append(
        {
            "model": m["model"],
            "report": m["report"],
            "cf_matrix": m["cf_matrix"],
            "f1": m["f1_score"],
            "train_time": m["train_time"],
        }
    )

spyder_eye(best_models, models)

Training set of models...:   0%|          | 0/2 [00:00<?, ?it/s]

ERROR 1: TIFFReadEncodedStrip:Read error at scanline 4294967295; got 6048 bytes, expected 21960
ERROR 1: TIFFReadEncodedStrip() failed.
ERROR 1: /Users/stephenhawking/Coding/ML/low2high_map/src/../data/processing/resized/labels/2_comp_s2_90d_b2_l2a_med.frag.0.tif, band 1: IReadBlock failed at X offset 0, Y offset 3948: TIFFReadEncodedStrip() failed.


count of value in secure mask:  68659627


Training set of models...:  50%|█████     | 1/2 [00:30<00:30, 30.52s/it]


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (15,) + inhomogeneous part.

In [5]:
print(best_models)

[{'mask_mode': 'random', 'r': 1, 'percent': 0.01, 'homogen_percent': 1, 'resize': 'by_sign', 'stratify': True, 'classes': array([10594, 10594, 10594, 10594, 10594, 10594]), 'report': '              precision    recall  f1-score   support\n\n           1       0.81      0.84      0.83      2093\n           2       0.64      0.68      0.66      2170\n           3       0.48      0.61      0.54      2124\n           4       0.68      0.61      0.64      2066\n           5       0.56      0.55      0.55      2120\n           6       0.47      0.36      0.41      2140\n\n    accuracy                           0.61     12713\n   macro avg       0.61      0.61      0.60     12713\nweighted avg       0.61      0.61      0.60     12713\n', 'cf_matrix': array([[1762,   28,  174,   42,   43,   44],
       [  99, 1470,  399,   23,   50,  129],
       [ 150,  321, 1293,   72,  107,  181],
       [  73,  123,  216, 1251,  212,  191],
       [  37,  145,  201,  267, 1160,  310],
       [  51,  227,  

In [ ]:
do_create_map = True
if do_create_map:
    # Получаем эталоны
    etalons = DEFAULT_PATH["etalons"]
    map_name = "dw"
    etalons_list = parse_tifs_from(etalons, typeof="etalon", verbose=False)
    etalons_list = etalons_list.query(f"year == {year} and map =='{map_name}'")[
        "path"
    ].to_list()

    # Загружаем тайлы с учетом quality
    quality = True
    if quality:
        tiles = DEFAULT_PATH["images"] + "tiles/"
        tiles = parse_tifs_from(tiles, typeof="tile", verbose=True)
        tiles = tiles.query(f"year == {year} and band in @only_bands").sort_values(
            "band"
        )["path"]
        tiles = tiles.to_list()
        loaded_tiles = []
        for t in tiles:
            loaded_tiles.append(load_tif(t, only_first=True, verbose=False))
    else:
        _, _, loaded_tiles, _ = generate_dataset(
            signs_paths, labels_paths, percent=0.0001, verbose=False
        )

    # Сортируем модели по f1_score
    sorted_models = sorted(
        zip(best_models, models), key=lambda x: x[0]["f1_score"], reverse=True
    )[:1]

    predicted_res = []
    validation_res = []
    for idx, (bm, model_dict) in enumerate(sorted_models):
        m = model_dict["model"]
        mm = bm["mask_mode"]
        f1 = bm["f1_score"]
        pc = bm["percent"]
        tt = bm["train_time"]
        out = DEFAULT_PATH["output"] + f"auto_10m_{f1:.3f}_{mm}_{pc}_{tt}.tif"
        pred_map = create_map(loaded_tiles, m, out)
        predicted_res.append(pred_map)
        val_result = validate_how_tif(pred_map, etalons_list)
        validation_res.append({"model_idx": idx, "validation": val_result})

    # Визуализация: столбцы по убыванию f1_score
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, len(predicted_res), figsize=(20, 5))
    # Если только одна модель, axes не является списком, а объектом Axes
    if len(predicted_res) == 1:
        axes = [axes]
    for i, pred_map in enumerate(predicted_res):
        # Предполагается, что pred_map - numpy array
        axes[i].imshow(pred_map)
        axes[i].set_title(f"Model {i + 1}\nf1={sorted_models[i][0]['f1_score']:.3f}")
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()


Load 'tile' from cached DataFrame to path:  /Users/stephenhawking/Coding/ML/low2high_map/src/../data/processing/path2tif_tile.csv


AttributeError: 'list' object has no attribute 'sort_values'